# Stage 2: Data Preparation, EDA and Preprocessing

Consumes the Stage 1 null reference panel: risk set, features, the loan-disjoint + observation-time-blocked split, frozen preprocessing fitted on DEVELOPMENT only, the 35-candidate grid, dev-only EDA, macro collinearity and a UK plausibility table. The frozen `preprocessing_meta.json` is what the Stage 3 runner applies to every panel.

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
# the DGP module, holds the simulator and the true hazard
import credit_dgp as dgp
# shared helpers used across every stage notebook
from pipeline_utils import (add_derived, save_json, load_json, Z_SOURCES,
    time_blocked_loan_disjoint_split, EMP_DUMMIES, HOUSING_LEVELS, HOUSING_BASELINE,
    EVIDENCE_LEVELS, EVIDENCE_BASELINE, PURPOSE_LEVELS, PURPOSE_BASELINE,
    ADVERSE_LEVELS, ADVERSE_BASELINE, REGION_LEVELS, REGION_BASELINE, AGE_BANDS, AGE_BASELINE)
# plot and display settings for this notebook
plt.rcParams.update({'figure.figsize':(10,3.5),'axes.grid':True,'grid.color':'#e1e0d9','axes.titlesize':10})
pd.set_option('display.width',200,'display.max_columns',60)
# folder where every result file gets saved
OUT=Path('outputs')
# the public 35-pair candidate grid, names only, no truth labels
registry=load_json(OUT/'registry_public.json')['candidate_pairs']
# seeds, strengths and split boundaries saved by Stage 1
manifest=load_json(OUT/'experiment_manifest.json')
# the null-condition reference panel Stage 1 wrote
panel=pd.read_csv('synthetic_credit_panel.csv')
print('loaded reference panel:', panel.shape, '| candidate pairs', len(registry))

## 2.1 Modelling risk set (active rows incl. 1-89 DPD; Current-only is a sensitivity)

In [ ]:
# add the affordability ratios and other derived columns
pan=add_derived(panel)
# active (non-terminal) rows with an observable next-quarter target; terminal rows are NaN
mdl=pan[pan['default_next_quarter'].notna()].copy()
# target as a clean 0/1 integer
mdl['default_next_quarter']=mdl['default_next_quarter'].astype(int)
print(f"risk set: {len(mdl):,} rows, {mdl['loan_id'].nunique():,} loans; "
      f"pre-default arrears rows included = {(mdl['performance_status_t']!='Current').mean():.1%}")

## 2.3/2.4 Loan-disjoint AND observation-time-blocked split

In [ ]:
# assign each row to dev, val or conf using the fixed time-blocked, loan-disjoint split
mdl['part']=time_blocked_loan_disjoint_split(mdl, manifest['split']['dev_end'], manifest['split']['val_end'])
# drop rows that fall outside all three blocks
mdl=mdl[mdl['part'].notna()].copy()
rows=[]
# build one summary row per partition: row count, loan count, date range, default rate
for pt in ['dev','val','conf']:
    s=mdl[mdl['part']==pt]
    rows.append({'partition':pt,'rows':len(s),'loans':s['loan_id'].nunique(),
                 'obs_from':str(s['observation_quarter'].min()),'obs_to':str(s['observation_quarter'].max()),
                 'default_rate_%':round(s['default_next_quarter'].mean()*100,3)})
display(pd.DataFrame(rows))
save_json(rows, OUT/'partition_summary.json')
import itertools
# check no loan appears in more than one partition
L={pt:set(mdl[mdl['part']==pt]['loan_id']) for pt in ['dev','val','conf']}
assert not any(L[a]&L[b] for a,b in itertools.combinations(L,2)), 'partitions share loans'
print('loan-disjoint across partitions: OK')
# development rows only, used to freeze the scaler below
dev=mdl[mdl['part']=='dev'].copy()

## 2.5 Preprocessing (encodings + standardisation; fitted on dev data only)

In [ ]:
# reference-coded dummies on the whole modelling frame
for lvl in EMP_DUMMIES: mdl[f'emp_{lvl}']=(mdl['employment_status_at_origination']==lvl).astype(int)
# one dummy per housing level except the baseline
for lvl in HOUSING_LEVELS:
    if lvl!=HOUSING_BASELINE: mdl[f'housing_{lvl}']=(mdl['housing_status']==lvl).astype(int)
# one dummy per income-evidence level except the baseline
for lvl in EVIDENCE_LEVELS:
    if lvl!=EVIDENCE_BASELINE: mdl[f'evid_{lvl}']=(mdl['income_evidence_type']==lvl).astype(int)
# one dummy per loan-purpose level except the baseline
for lvl in PURPOSE_LEVELS:
    if lvl!=PURPOSE_BASELINE: mdl[f'purpose_{lvl}']=(mdl['loan_purpose']==lvl).astype(int)
# one dummy per adverse-record level except the baseline
for lvl in ADVERSE_LEVELS:
    if lvl!=ADVERSE_BASELINE: mdl[f'adverse_{lvl}']=(mdl['public_adverse_record']==lvl).astype(int)
# one dummy per region except the baseline
for lvl in REGION_LEVELS:
    if lvl!=REGION_BASELINE: mdl[f'region_{lvl}']=(mdl['uk_region']==lvl).astype(int)
# one dummy per age band except the baseline
for lvl in AGE_BANDS:
    if lvl!=AGE_BASELINE: mdl[f'age_{lvl}']=(mdl['age_band']==lvl).astype(int)
# collect the dummy column names for each categorical group
emp_d=[f'emp_{l}' for l in EMP_DUMMIES]; housing_d=[f'housing_{l}' for l in HOUSING_LEVELS if l!=HOUSING_BASELINE]
evid_d=[f'evid_{l}' for l in EVIDENCE_LEVELS if l!=EVIDENCE_BASELINE]; purpose_d=[f'purpose_{l}' for l in PURPOSE_LEVELS if l!=PURPOSE_BASELINE]
adverse_d=[f'adverse_{l}' for l in ADVERSE_LEVELS if l!=ADVERSE_BASELINE]; region_d=[f'region_{l}' for l in REGION_LEVELS if l!=REGION_BASELINE]
age_d=[f'age_{l}' for l in AGE_BANDS if l!=AGE_BASELINE]
# current arrears state at entry (loan began the quarter in 1-89 DPD) as a binary predictor,
# matching the DGP beta_inarrears hazard term. Read off the stored entry-state performance_status_t.
mdl['in_arrears_t']=mdl['performance_status_t'].isin(['1-29 DPD','30-59 DPD','60-89 DPD']).astype(int)
# frozen scaler fitted on DEV rows only
dev=mdl[mdl['part']=='dev']
# mean utilisation on dev, used to fill missing values below
util_impute=float(dev['revolving_utilisation'].mean())
mdl['revolving_utilisation']=mdl['revolving_utilisation'].fillna(util_impute)
scaler={}
# z-score every driver using the dev-only mean and sd, then keep those moments for later stages
for zc,src in Z_SOURCES.items():
    mu,sd=float(dev[src].mean()),float(dev[src].std(ddof=0)); scaler[zc]={'mu':mu,'sd':sd if sd>0 else 1.0}
    mdl[zc]=(mdl[src]-mu)/scaler[zc]['sd']
# composite macro stress index on the standardised scale
mdl['z_M']=0.5*mdl['z_unemp']+0.25*mdl['z_cpi']+0.25*mdl['z_rate']
# build every candidate interaction column as micro z-score times macro z-score
for pr in registry: mdl[pr['col']]=mdl[pr['micro']].values*mdl[pr['macro']].values
print('encoded + standardised on DEV moments; util_impute=',round(util_impute,3))

## 2.6 Feature lists + frozen meta + reference exports

In [ ]:
# continuous micro drivers that get z-scored
micro_z_base=['z_income','z_dti','z_dsr','z_pti','z_residual','z_savbuf','z_lti','z_util','z_hist','z_tenure','z_mob']
# raw count-style micro features, used as is
micro_raw=['delinquency_count_24m','recent_credit_searches_6m','financial_dependants']
# continuous macro drivers that get z-scored
macro_z_base=['z_unemp','z_cpi','z_rate','z_gdp','z_wage','z_dsr_m','z_credit_m','z_sav_m']
# extra static and dynamic features added to the schema
v6_new=['income_shock_t','prior_arrears_count_12m_t','in_arrears_t','new_credit_accounts_24m','open_credit_accounts','other_household_earner_flag','existing_customer_flag','util_missing']
# every categorical dummy column, all groups combined
cat_d=emp_d+housing_d+evid_d+purpose_d+adverse_d+region_d+age_d
# the full base feature list every model trains on
BASE_FEATURES=micro_z_base+micro_raw+v6_new+cat_d+macro_z_base
# the 35 interaction columns
IX=[p['col'] for p in registry]
# fail loudly if any feature has a non-finite value
assert np.isfinite(mdl[BASE_FEATURES+IX].values).all(), 'non-finite feature'
# the frozen preprocessing recipe later stages load and reuse
meta={'random_state':42,'scaler':scaler,'util_impute':util_impute,'base_features':BASE_FEATURES,
      'candidate_pairs':registry,'micro_z_smooth':micro_z_base,
      'macro_z_smooth':['z_unemp','z_cpi','z_rate','z_gdp','z_wage'],
      'housing_levels':HOUSING_LEVELS,'housing_baseline':HOUSING_BASELINE,
      'region_levels':REGION_LEVELS,'region_baseline':REGION_BASELINE,
      'split':manifest['split'],'n_orig_per_quarter':manifest['n_orig_per_quarter']}
save_json(meta, OUT/'preprocessing_meta.json')
# id and label columns kept alongside the model features in the export
id_cols=['loan_id','origination_quarter','observation_quarter','part','default_next_quarter','default_next_12m','performance_status_t','quarters_on_book','months_on_book']
# raw macro levels kept for reference, not used directly by the models
raw_macro=['unemployment','cpi_inflation','bank_rate','gdp_growth','wage_growth','dsr','credit_growth','saving_ratio']
export_cols=id_cols+raw_macro+BASE_FEATURES+IX
# save dev, val and conf as their own model-ready csv files
for pt in ['dev','val','conf']:
    mdl[mdl['part']==pt][export_cols].to_csv(OUT/f'reference_{pt}_model_ready.csv',index=False)
print(f'meta saved: {len(BASE_FEATURES)} base features + {len(IX)} interaction columns; reference dev/val/conf exported.')

## 2.7 Development-only EDA

In [ ]:
# outcome-based EDA restricted to DEVELOPMENT data (blind to any hidden interaction: null panel)
fig,ax=plt.subplots(1,2,figsize=(12,3.5))
# average default rate by quarter, left panel
q=dev.groupby('observation_quarter')['default_next_quarter'].mean()
ax[0].plot(range(len(q)), q.values*100, color='#2a78d6'); ax[0].set_title('dev: next-q default rate by quarter (%)'); ax[0].set_xlabel('quarter index')
# average default rate by employment status, right panel
seg=dev.groupby('employment_status_at_origination')['default_next_quarter'].mean().sort_values()*100
ax[1].barh(seg.index, seg.values, color='#1baf7a'); ax[1].set_title('dev: default rate by employment (%)')
plt.tight_layout(); plt.show()

## 2.8 Macro collinearity (correlation + condition number)

In [ ]:
# the eight standardised macro drivers
mac=['z_unemp','z_cpi','z_rate','z_gdp','z_wage','z_dsr_m','z_credit_m','z_sav_m']
# one row per quarter on dev only, so the correlation is not inflated by repeated quarters
qmac=mdl[mdl['part']=='dev'].drop_duplicates('observation_quarter')[mac]
cor=qmac.corr()
display(cor.round(2))
# condition number from the eigenvalues, a high number means the macro axes are hard to tell apart
ev=np.linalg.eigvalsh(cor.values); cond=float(np.sqrt(ev.max()/max(ev.min(),1e-9)))
print(f'condition number of the macro correlation matrix = {cond:.1f} (high -> attribution across correlated macro axes is hard)')

## 2.9 data plausibility table (generated vs reference anchors)

In [ ]:
plaus=[]
# one row per loan, so each borrower counts once
loans=pan.drop_duplicates('loan_id')
# compare the generated population against a few real UK reference anchors
plaus.append({'quantity':'full-time median gross income','generated':round(float(loans.loc[loans['employment_status_at_origination']=='full_time','gross_annual_income'].median()),0),'reference_anchor':'ONS ASHE ~35,000'})
plaus.append({'quantity':'self-employed share','generated':round(float((loans['employment_status_at_origination']=='self_employed').mean()),3),'reference_anchor':'~0.12 (UK working-age)'})
plaus.append({'quantity':'London region share','generated':round(float((loans['uk_region']=='LON').mean()),3),'reference_anchor':'ONS ~0.13'})
plaus.append({'quantity':'mortgage housing share','generated':round(float((loans['housing_status']=='mortgage').mean()),3),'reference_anchor':'~0.30'})
plaus.append({'quantity':'mean next-q default rate','generated':round(float(mdl['default_next_quarter'].mean()),4),'reference_anchor':'~0.006 booked unsecured'})
display(pd.DataFrame(plaus))
print('Differences from anchors are documented, not forced; distributions are UK-informed, not Lloyds-representative.')